# 02 · Qwen2.5 升级：LoRA / QLoRA（模型尺寸提升）

**模型**：Qwen2.5-0.5B-Instruct（可全参 SFT / LoRA / QLoRA）、Qwen2.5-1.5B-Instruct（LoRA / QLoRA）
**显存策略**：0.5B 全参 SFT 可行；1.5B 仅 LoRA/QLoRA。

In [ ]:
# [0] GPU 自检：确认拿到 T4，并钉死 T4 兼容参数
import torch
assert torch.cuda.is_available(), "未分配到 GPU！请 运行时->更改运行时类型 选 T4 GPU"
name = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {name} | VRAM: {vram:.1f} GB | CC: {torch.cuda.get_device_capability(0)}")
# T4 = sm7.5：无原生 bf16、无 FlashAttention-2 -> 统一 fp16 + sdpa
DTYPE, ATTN = "float16", "sdpa"
print("钉死参数: dtype=%s attn=%s (T4 无 bf16/FA2)" % (DTYPE, ATTN))


In [ ]:
# [1] 安装依赖（T4 / CUDA12 兼容钉死版本）
# 关键：transformers 必须 <5（trl 0.x 与 5.x 不兼容）；不装 flash-attn（T4 sm7.5 不支持）
!pip install -q "torch==2.3.1" "transformers==4.44.2" "datasets==2.20.0" "peft==0.12.0" \
    "trl==0.9.6" "accelerate==0.33.0" "bitsandbytes==0.43.3" "sentencepiece" "tiktoken" \
    "numpy<2" "pandas" "matplotlib"
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)


In [ ]:
# [2] 克隆仓库并安装 minillm 包（单一事实源）
import os, subprocess
if not os.path.exists("cs0502-mini-llm"):
    subprocess.run(["git", "clone", "--depth", "1", "https://cnb.cool/sizhezhang/cs0502-mini-llm.git"], check=True)
os.chdir("cs0502-mini-llm")
!pip install -q -e .
import minillm
from minillm.config import ExpConfig
from minillm import data as D
from minillm import evaluate as E
from minillm import train as T
from minillm import quantize as Q
from minillm import bench as B
from minillm.record import Recorder
print("minillm", minillm.__version__, "| cwd", os.getcwd())


In [ ]:
# [3] 挂载 Google Drive，统一结果/数据落盘路径（Colab 文件系统易失，必须落 Drive）
from google.colab import drive
drive.mount("/content/drive")
import os
BASE = "/content/drive/MyDrive/cs0502-mini-llm"
DATA = os.path.join(BASE, "data"); RESULTS = os.path.join(BASE, "results")
os.makedirs(DATA, exist_ok=True); os.makedirs(RESULTS, exist_ok=True)
rec = Recorder(RESULTS)
print("DATA:", DATA, "| RESULTS:", RESULTS)


In [ ]:
# 结果打表辅助
import pandas as pd
def show(df, title=""):
    if title: print("\n==", title, "==")
    from IPython.display import display
    display(df)
    return df


In [ ]:
import torch as _t
def _ft(d): return {'float16':_t.float16,'bfloat16':_t.bfloat16,'float32':_t.float32}[d]
import pandas as pd

In [ ]:
# 基线（未微调）评测：qwen2.5-0.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-0.5b-courseqa-baseline", model_key="qwen2.5-0.5b", task="courseqa",
                method="none", dtype=DTYPE, attn=ATTN)
model, tok = E.load_model(cfg)
m_base, ps_base = E.evaluate(model, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_base, ps_base)
del model
import torch as _t; _t.cuda.empty_cache()
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_base.items() if isinstance(v,(int,float))}}]), "BASELINE qwen2.5-0.5b/courseqa")


In [ ]:
# 微调（lora）+ 评测：qwen2.5-0.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-0.5b-courseqa-lora", model_key="qwen2.5-0.5b", task="courseqa",
                method="lora", use_4bit=false,
                dtype=DTYPE, attn=ATTN, epochs=2, batch_size=4, grad_accum=4, lr=2e-4)
out_dir = os.path.join(RESULTS, "ckpt", cfg.run_id)
T.train(D.load_dataset("courseqa", DATA), "courseqa", cfg, out_dir)
# 加载微调后模型评测
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base, tok = E.load_model(ExpConfig(run_id="tmp", model_key="qwen2.5-0.5b", dtype=DTYPE, attn=ATTN))
if "lora" != "sft":
    base = PeftModel.from_pretrained(base, os.path.join(out_dir, "final"))
    base = base.merge_and_unload()
else:
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out_dir, "final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN)
base.eval()
m_ft, ps_ft = E.evaluate(base, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_ft, ps_ft)
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_ft.items() if isinstance(v,(int,float))}}]), "FINETUNED lora")
del base; _t.cuda.empty_cache()


In [ ]:
# 基线（未微调）评测：qwen2.5-0.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-0.5b-courseqa-baseline", model_key="qwen2.5-0.5b", task="courseqa",
                method="none", dtype=DTYPE, attn=ATTN)
model, tok = E.load_model(cfg)
m_base, ps_base = E.evaluate(model, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_base, ps_base)
del model
import torch as _t; _t.cuda.empty_cache()
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_base.items() if isinstance(v,(int,float))}}]), "BASELINE qwen2.5-0.5b/courseqa")


In [ ]:
# 微调（qlora）+ 评测：qwen2.5-0.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-0.5b-courseqa-qlora", model_key="qwen2.5-0.5b", task="courseqa",
                method="qlora", use_4bit=true,
                dtype=DTYPE, attn=ATTN, epochs=2, batch_size=4, grad_accum=4, lr=2e-4)
out_dir = os.path.join(RESULTS, "ckpt", cfg.run_id)
T.train(D.load_dataset("courseqa", DATA), "courseqa", cfg, out_dir)
# 加载微调后模型评测
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base, tok = E.load_model(ExpConfig(run_id="tmp", model_key="qwen2.5-0.5b", dtype=DTYPE, attn=ATTN))
if "qlora" != "sft":
    base = PeftModel.from_pretrained(base, os.path.join(out_dir, "final"))
    base = base.merge_and_unload()
else:
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out_dir, "final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN)
base.eval()
m_ft, ps_ft = E.evaluate(base, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_ft, ps_ft)
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_ft.items() if isinstance(v,(int,float))}}]), "FINETUNED qlora")
del base; _t.cuda.empty_cache()


In [ ]:
# 基线（未微调）评测：qwen2.5-0.5b / jsonout
cfg = ExpConfig(run_id="qwen2.5-0.5b-jsonout-baseline", model_key="qwen2.5-0.5b", task="jsonout",
                method="none", dtype=DTYPE, attn=ATTN)
model, tok = E.load_model(cfg)
m_base, ps_base = E.evaluate(model, tok, D.load_dataset("jsonout", DATA), "jsonout", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_base, ps_base)
del model
import torch as _t; _t.cuda.empty_cache()
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_base.items() if isinstance(v,(int,float))}}]), "BASELINE qwen2.5-0.5b/jsonout")


In [ ]:
# 微调（lora）+ 评测：qwen2.5-0.5b / jsonout
cfg = ExpConfig(run_id="qwen2.5-0.5b-jsonout-lora", model_key="qwen2.5-0.5b", task="jsonout",
                method="lora", use_4bit=false,
                dtype=DTYPE, attn=ATTN, epochs=2, batch_size=4, grad_accum=4, lr=2e-4)
out_dir = os.path.join(RESULTS, "ckpt", cfg.run_id)
T.train(D.load_dataset("jsonout", DATA), "jsonout", cfg, out_dir)
# 加载微调后模型评测
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base, tok = E.load_model(ExpConfig(run_id="tmp", model_key="qwen2.5-0.5b", dtype=DTYPE, attn=ATTN))
if "lora" != "sft":
    base = PeftModel.from_pretrained(base, os.path.join(out_dir, "final"))
    base = base.merge_and_unload()
else:
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out_dir, "final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN)
base.eval()
m_ft, ps_ft = E.evaluate(base, tok, D.load_dataset("jsonout", DATA), "jsonout", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_ft, ps_ft)
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_ft.items() if isinstance(v,(int,float))}}]), "FINETUNED lora")
del base; _t.cuda.empty_cache()


In [ ]:
# 基线（未微调）评测：qwen2.5-0.5b / intent
cfg = ExpConfig(run_id="qwen2.5-0.5b-intent-baseline", model_key="qwen2.5-0.5b", task="intent",
                method="none", dtype=DTYPE, attn=ATTN)
model, tok = E.load_model(cfg)
m_base, ps_base = E.evaluate(model, tok, D.load_dataset("intent", DATA), "intent", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_base, ps_base)
del model
import torch as _t; _t.cuda.empty_cache()
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_base.items() if isinstance(v,(int,float))}}]), "BASELINE qwen2.5-0.5b/intent")


In [ ]:
# 微调（lora）+ 评测：qwen2.5-0.5b / intent
cfg = ExpConfig(run_id="qwen2.5-0.5b-intent-lora", model_key="qwen2.5-0.5b", task="intent",
                method="lora", use_4bit=false,
                dtype=DTYPE, attn=ATTN, epochs=2, batch_size=4, grad_accum=4, lr=2e-4)
out_dir = os.path.join(RESULTS, "ckpt", cfg.run_id)
T.train(D.load_dataset("intent", DATA), "intent", cfg, out_dir)
# 加载微调后模型评测
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base, tok = E.load_model(ExpConfig(run_id="tmp", model_key="qwen2.5-0.5b", dtype=DTYPE, attn=ATTN))
if "lora" != "sft":
    base = PeftModel.from_pretrained(base, os.path.join(out_dir, "final"))
    base = base.merge_and_unload()
else:
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out_dir, "final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN)
base.eval()
m_ft, ps_ft = E.evaluate(base, tok, D.load_dataset("intent", DATA), "intent", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_ft, ps_ft)
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_ft.items() if isinstance(v,(int,float))}}]), "FINETUNED lora")
del base; _t.cuda.empty_cache()


In [ ]:
# 基线（未微调）评测：qwen2.5-1.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-1.5b-courseqa-baseline", model_key="qwen2.5-1.5b", task="courseqa",
                method="none", dtype=DTYPE, attn=ATTN)
model, tok = E.load_model(cfg)
m_base, ps_base = E.evaluate(model, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_base, ps_base)
del model
import torch as _t; _t.cuda.empty_cache()
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_base.items() if isinstance(v,(int,float))}}]), "BASELINE qwen2.5-1.5b/courseqa")


In [ ]:
# 微调（qlora）+ 评测：qwen2.5-1.5b / courseqa
cfg = ExpConfig(run_id="qwen2.5-1.5b-courseqa-qlora", model_key="qwen2.5-1.5b", task="courseqa",
                method="qlora", use_4bit=true,
                dtype=DTYPE, attn=ATTN, epochs=2, batch_size=4, grad_accum=4, lr=2e-4)
out_dir = os.path.join(RESULTS, "ckpt", cfg.run_id)
T.train(D.load_dataset("courseqa", DATA), "courseqa", cfg, out_dir)
# 加载微调后模型评测
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
base, tok = E.load_model(ExpConfig(run_id="tmp", model_key="qwen2.5-1.5b", dtype=DTYPE, attn=ATTN))
if "qlora" != "sft":
    base = PeftModel.from_pretrained(base, os.path.join(out_dir, "final"))
    base = base.merge_and_unload()
else:
    base = AutoModelForCausalLM.from_pretrained(os.path.join(out_dir, "final"), torch_dtype=_ft(DTYPE), attn_implementation=ATTN)
base.eval()
m_ft, ps_ft = E.evaluate(base, tok, D.load_dataset("courseqa", DATA), "courseqa", cfg)
rec.log_run(cfg.run_id, cfg.to_dict(), m_ft, ps_ft)
show(pd.DataFrame([{"run": cfg.run_id, **{k: round(v,4) for k,v in m_ft.items() if isinstance(v,(int,float))}}]), "FINETUNED qlora")
del base; _t.cuda.empty_cache()


In [ ]:
runs = rec.load_runs()
df = pd.DataFrame([{"run": r["run_id"], "model": r["config"]["model_key"], "task": r["config"]["task"],
                    "method": r["config"]["method"],
                    **{k: round(v,4) for k,v in r["metrics"].items() if isinstance(v,(int,float)) and k in ("overall","exact_match","token_f1","accuracy","macro_f1","schema_valid_rate")}}
                   for r in runs if r["config"]["model_key"].startswith("qwen")])
show(df, "QWEN RUNS")
